**1. Introduction and Objective**

**Problem:** Traditional Applicant Tracking Systems (ATS) rely heavily on exact keyword matching, causing recruiters to miss highly qualified candidates who use different terminology or possess strong implicit skills.

**Dataset:** The system utilizes a dataset of unstructured or semi-structured Job Descriptions (JDs) and candidate profiles (resumes, work history).

**Goal:** Build a production-grade, two-stage Hybrid AI Candidate Ranking System. The model will use an open-source Bi-encoder (BAAI/bge-m3) for initial dense semantic retrieval, followed by a Cross-encoder (Qwen/Qwen3-Reranker) for deep contextual scoring, evaluating candidates on true fit rather than keyword presence.

**2. Data Loading and Exploration**

Since your data is in Google Drive, if you are using Google Colab, you will first need to mount your drive. We will use pandas to read the .jsonl file and python-docx to extract text from the .docx files.

In [1]:
!pip install python-docx
import pandas as pd
import json
import docx
import os



# Helper to find the files if the path is wrong
print("Searching for files in Database...")
for root, dirs, files in os.walk("/kaggle/input/datasets/vivekron/india-runs-data-and-ai-challenge"):
    if "candidates.jsonl" in files:
        print(f"Found candidates.jsonl at: {os.path.join(root, 'candidates.jsonl')}")
        # Auto-update BASE_PATH if found
        BASE_PATH = root
        break
else:
    # Fallback to current path if search fails
    BASE_PATH = "/content/drive/MyDrive/[PUB] India_runs_data_and_ai_challenge/[PUB] India_runs_data_and_ai_challenge"

def read_docx(file_path):
    """Extracts text from a .docx file."""
    if not os.path.exists(file_path):
        print(f"Warning: File not found at {file_path}")
        return ""
    try:
        doc = docx.Document(file_path)
        return "\n".join([para.text for para in doc.paragraphs if para.text.strip()])
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        return ""

print(f"Using BASE_PATH: {BASE_PATH}")

# Load Candidates
candidates_path = os.path.join(BASE_PATH, "candidates.jsonl")
if os.path.exists(candidates_path):
    candidates_df = pd.read_json(candidates_path, lines=True)
    print(f"Successfully loaded {len(candidates_df)} candidates.")
    print("Candidate Data Columns:", candidates_df.columns.tolist())
else:
    print(f"Error: Still could not find candidates.jsonl. Please check the path.")

# Load Job Description
jd_path = os.path.join(BASE_PATH, "job_description.docx")
jd_text = read_docx(jd_path)
print(f"Job Description Length: {len(jd_text)} characters.")

# Load Signals
signals_path = os.path.join(BASE_PATH, "redrob_signals_doc.docx")
signals_text = read_docx(signals_path)

Searching for files in Database...
Found candidates.jsonl at: /kaggle/input/datasets/vivekron/india-runs-data-and-ai-challenge/[PUB] India_runs_data_and_ai_challenge/[PUB] India_runs_data_and_ai_challenge/India_runs_data_and_ai_challenge/candidates.jsonl
Using BASE_PATH: /kaggle/input/datasets/vivekron/india-runs-data-and-ai-challenge/[PUB] India_runs_data_and_ai_challenge/[PUB] India_runs_data_and_ai_challenge/India_runs_data_and_ai_challenge
Successfully loaded 100000 candidates.
Candidate Data Columns: ['candidate_id', 'profile', 'career_history', 'education', 'skills', 'certifications', 'languages', 'redrob_signals']
Job Description Length: 9564 characters.


**3. Data Preprocessing**

Because candidates.jsonl likely contains nested data (e.g., lists of dictionaries for education and experience), we must flatten this into a dense "Profile Summary" string so the embedding model can understand the candidate's holistic background.

In [2]:
def flatten_candidate_profile(row):
    """
    Dynamically converts a nested JSON row into a single semantic string.
    This ensures all implicit skills and experience are captured for the embedding model.
    """
    profile_parts = []

    # We iterate through whatever columns the candidate_schema.json dictates
    for col in row.index:
        val = row[col]

        # Fix: Safely check for nulls and empty structures
        if pd.isna(val) if isinstance(val, (str, float, int)) else False:
            continue
        if val is None or (isinstance(val, (list, dict, str)) and len(val) == 0):
            continue

        # If the field is a list or dict (like experience history), convert to string
        if isinstance(val, (list, dict)):
            profile_parts.append(f"{col.upper()}: {json.dumps(val)}")
        else:
            profile_parts.append(f"{col.upper()}: {str(val)}")

    # Combine into a single text block and clean whitespace
    raw_text = " | ".join(profile_parts)
    return " ".join(raw_text.split())

print("Preprocessing Candidate Profiles...")
candidates_df['semantic_text'] = candidates_df.apply(flatten_candidate_profile, axis=1)

# Ensure we have an ID column
id_col = 'candidate_id' if 'candidate_id' in candidates_df.columns else candidates_df.columns[0]
print(f"Using '{id_col}' as the unique identifier.")

Preprocessing Candidate Profiles...
Using 'candidate_id' as the unique identifier.


**4. Model Building and Training**

We initialize the retrieval infrastructure. Since we are doing zero-shot ranking against a provided challenge JD, we don't "train" a model from scratch. Instead, we populate our vector database with the embedded candidate profiles.

In [3]:
!pip install qdrant-client
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams, PointStruct
import torch
import gc
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import time
import numpy as np

print("Initializing Open-Source ML Stack...")

# 1. Check for GPU availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using Compute Device: {device.upper()}")

# 2. Initialize Bi-Encoder
embedding_model = SentenceTransformer('BAAI/bge-m3', device=device)
# Correct way to set max length for SentenceTransformers
embedding_model.max_seq_length = 512

# 3. Setup Vector Database
client = QdrantClient(":memory:")
if client.collection_exists(collection_name="challenge_candidates"):
    client.delete_collection(collection_name="challenge_candidates")

client.create_collection(
    collection_name="challenge_candidates",
    vectors_config=VectorParams(size=1024, distance=Distance.COSINE),
)

# 4. Vectorize and Upsert Candidates
print("Generating Embeddings and Upserting to Qdrant...")
start_time = time.time()

candidate_texts = candidates_df['semantic_text'].tolist()
candidate_ids = candidates_df[id_col].astype(str).tolist()

# Smaller chunks and smaller batch_size to prevent OOM
chunk_size = 2500
for i in range(0, len(candidate_texts), chunk_size):
    batch_texts = candidate_texts[i:i + chunk_size]
    batch_ids = candidate_ids[i:i + chunk_size]

    # Generate embeddings (max_length removed from here as it is set on the model above)
    batch_embeddings = embedding_model.encode(
        batch_texts,
        batch_size=32,
        show_progress_bar=True,
        normalize_embeddings=True
    )

    # Prepare points
    points = [
        PointStruct(
            id=i + j,
            vector=emb.tolist(),
            payload={"candidate_id": batch_ids[j], "text": batch_texts[j]}
        )
        for j, emb in enumerate(batch_embeddings)
    ]

    # Upsert
    client.upsert(collection_name="challenge_candidates", points=points)

    # Force memory cleanup
    del batch_embeddings
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()

    if (i + chunk_size) % 10000 == 0 or (i + chunk_size) >= len(candidate_texts):
        print(f"Processed {min(i + chunk_size, len(candidate_texts))}/{len(candidate_texts)} candidates...")

end_time = time.time()
print(f"Pipeline completed in {(end_time - start_time) / 60:.2f} minutes.")

# 5. Initialize Cross-Encoder
print("Loading Cross-encoder...")
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-Reranker-0.6B")
reranker_model = AutoModelForSequenceClassification.from_pretrained("Qwen/Qwen3-Reranker-0.6B")
reranker_model.to(device).eval()
print("Cross-encoder ready.")

Initializing Open-Source ML Stack...
Using Compute Device: CUDA


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Generating Embeddings and Upserting to Qdrant...


Batches:   0%|          | 0/79 [00:00<?, ?it/s]

Batches:   0%|          | 0/79 [00:00<?, ?it/s]

...

chat_template.jinja:   0%|          | 0.00/741 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Qwen3ForSequenceClassification LOAD REPORT from: Qwen/Qwen3-Reranker-0.6B
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Cross-encoder ready.


**5. Evaluation (Generating the Submission)**

Instead of testing against a known set, the goal of this challenge is to generate a ranked list. We will query our pipeline with the job_description.docx text, score all candidates, and format the output to match what sample_submission.csv likely expects.

In [10]:
import pandas as pd
import torch
import os

# =====================================================================
# CRITICAL FIX: Force the padding token configuration before inference
# =====================================================================
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
if reranker_model.config.pad_token_id is None:
    reranker_model.config.pad_token_id = tokenizer.pad_token_id
# =====================================================================

def generate_challenge_ranking(jd_text, top_k=100, rerank_batch_size=4):
    # Step 1: Broad Dense Retrieval
    jd_vector = embedding_model.encode(jd_text).tolist()
    
    search_response = client.query_points(
        collection_name="challenge_candidates",
        query=jd_vector,
        limit=top_k
    )
    
    retrieved = [hit.payload for hit in search_response.points]
    
    # Step 2: Deep Contextual Reranking (Memory-Safe Chunking)
    print(f"Reranking top {len(retrieved)} candidates with Cross-Encoder...")
    pairs = [[jd_text, cand['text']] for cand in retrieved]
    
    all_scores = []
    
    # Process the 100 pairs in small safe steps of 4
    for i in range(0, len(pairs), rerank_batch_size):
        batch_pairs = pairs[i:i + rerank_batch_size]
        
        with torch.no_grad():
            inputs = tokenizer(
                batch_pairs, 
                padding=True, 
                truncation=True, 
                return_tensors='pt', 
                max_length=512
            ).to(device)
            
            # Generate scores for this specific small batch
            batch_scores = reranker_model(**inputs, return_dict=True).logits.view(-1, ).float()
            all_scores.extend(batch_scores.cpu().tolist())
            
        # Keep the GPU memory pristine
        del inputs, batch_scores
        if device == "cuda":
            torch.cuda.empty_cache()
            
    # Assign accumulated scores back to candidates
    for i, cand in enumerate(retrieved):
        cand['ai_fit_score'] = all_scores[i]
        
    # Sort descending by the AI's confidence score
    ranked = sorted(retrieved, key=lambda x: x['ai_fit_score'], reverse=True)
    return ranked

print("Executing AI Ranking Pipeline against Job Description...")
# Enhance the JD query with the signals document if needed
combined_query = jd_text + "\n\nAdditional Signals:\n" + signals_text

# Run with a safe mini-batch size of 4 for the reranker to prevent OOM errors
final_ranking = generate_challenge_ranking(combined_query, top_k=100, rerank_batch_size=4)

# Format for Submission
submission_data = []
for rank, candidate in enumerate(final_ranking, start=1):
    submission_data.append({
        'candidate_id': candidate['candidate_id'],
        'rank': rank,
        'score': candidate['ai_fit_score']
    })

submission_df = pd.DataFrame(submission_data)

# --- THE FIX ---
# Save directly to Kaggle's writable working directory
submission_path = "/kaggle/working/my_submission.csv"
# ---------------

submission_df.to_csv(submission_path, index=False)

print(f"\nSuccess! Ranked {len(submission_df)} candidates.")
print(f"Submission saved to: {submission_path}")
print("\nTop 3 Candidates:")
print(submission_df.head(3))

Executing AI Ranking Pipeline against Job Description...
Reranking top 100 candidates with Cross-Encoder...

Success! Ranked 100 candidates.
Submission saved to: /kaggle/working/my_submission.csv

Top 3 Candidates:
   candidate_id  rank    score
0  CAND_0083670     1  6.87500
1  CAND_0053927     2  6.09375
2  CAND_0032640     3  5.93750
